# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [1]:
import os

# Limit libraries that use OpenMP (PyTorch, scikit-learn) to one thread each,
# so their separate OpenMP copies don't clash and crash or freeze the kernel
os.environ["OMP_NUM_THREADS"] = "1"

# Silences a related warning from the Hugging Face tokenizers library
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Encode all first sentences and all second sentences in two batches (faster than one by one)
first_sentences = [a for a, _ in sentence_pairs]
second_sentences = [b for _, b in sentence_pairs]
emb_first = model.encode(first_sentences)
emb_second = model.encode(second_sentences)

# Compute cosine similarity pair by pair
# [e1] and [e2] turn each 1D vector into 2D, which cosine_similarity requires
for (a, b), e1, e2 in zip(sentence_pairs, emb_first, emb_second):
    score = cosine_similarity([e1], [e2])[0][0]
    print(f"{score:.3f}  |  {a}  vs.  {b}")

W1027 13:37:23.426000 94899 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.522  |  A dog is playing in the park.  vs.  A dog is running in a field.
0.528  |  I love pizza.  vs.  I enjoy ice cream.
0.319  |  What is AI?  vs.  How does a computer learn?


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


### Results

| Pair | Cosine similarity |
|---|---|
| "I love pizza." vs. "I enjoy ice cream." | **0.528** |
| "A dog is playing in the park." vs. "A dog is running in a field." | **0.522** |
| "What is AI?" vs. "How does a computer learn?" | **0.319** |

### Which sentence pairs are the most semantically similar? Why?

The **pizza / ice cream** pair scored highest (0.528), with the **dog** pair right behind (0.522). The **AI** pair is clearly last (0.319).

**The top two are effectively tied.** A difference of 0.006 is far too small to call one pair "more similar" than the other; it's within the noise of small wording changes. The model sees both pairs as moderately and equally similar.

**Why the pizza pair scores so high even though it describes different foods:**
- The two sentences have an identical structure: *I + positive verb + food*.
- "Love" and "enjoy" are near-synonyms.
- Pizza and ice cream belong to the same category: popular, casual food.

The model encodes both as "someone expressing that they like a food", and at that level of abstraction they are close. A human would say they mean different things (two different preferences), but the model treats them as two examples of the same *kind* of statement.

**Why the dog pair doesn't score higher:** it shares the subject (a dog) and the frame ("A dog is ... in ..."), but two of its three content words change: *playing → running* and *park → field*. Each is related but distinct, and the model doesn't treat them as synonyms.

**Why the AI pair is lowest but still clearly above zero:** the sentences share no words, and they ask different things (a definition vs. a mechanism). A score of 0.319 still shows the model connects them through the shared topic of AI and machine learning. A Bag of Words approach would give this pair a similarity of exactly **0**, because there's no word overlap.

**Lesson:** this model rewards *matching structure and category* (pizza pair) about as much as *describing the same scene* (dog pair). A similarity score tells us two sentences are related in *some* way, not *how* they're related.

### Can you think of cases where cosine similarity might fail to capture true semantic meaning?

Our own results contain one: **the pizza / ice cream sentences score as high as the two dog sentences**, even though a human sees "two different preferences" vs. "the same dog doing nearly the same thing". Other common failure cases:

- **Negation:** "I love pizza" vs. "I don't love pizza" would likely score very high, since nearly every token and the topic are shared, yet the meanings are opposite.
- **Role reversal / word order:** "The dog bit the man" vs. "The man bit the dog" use the same words in different roles, so they get very similar embeddings despite describing different events.
- **Antonyms in the same context:** "The flight was cheap" vs. "The flight was expensive" are both about flight prices, so they sit close together.
- **Sarcasm:** "Great, another Monday" sounds positive on the surface but means the opposite.
- **Polysemy:** "python" (snake vs. programming language) and "bank" (river vs. money) can be misread when context is short.
- **Scores aren't absolute:** our highest score was only 0.528, which does not mean "53% similar". A score only has meaning compared with other scores from the same model; another model could give these pairs 0.8, 0.8 and 0.6 and rank them identically.

**Conclusion:** embeddings capture **topic, category and sentence structure** well, but they're weak at **logic**: negation, who did what to whom, and whether two statements actually say the same thing.

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [14]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# normalize_embeddings=True: every vector gets length 1, so the Euclidean distance
# that KMeans uses ranks documents the same way as cosine similarity
doc_embeddings = model.encode(documents, normalize_embeddings=True)
print(doc_embeddings.shape)   # (6, 384)

(6, 384)


In [15]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Compare several k values with the silhouette score (higher = better separated clusters)
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(doc_embeddings)
    print(f"k={k}  silhouette={silhouette_score(doc_embeddings, labels):.3f}")

# 2. Fit the final model
k = 3   # adjust after looking at the silhouette scores above
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(doc_embeddings)

k=2  silhouette=0.024
k=3  silhouette=0.029
k=4  silhouette=0.021
k=5  silhouette=0.010


In [16]:
# Print cluster assignments

#YOUR CODE HERE
import pandas as pd

results = pd.DataFrame({"document": documents, "cluster": clusters})

for cluster_id, group in results.groupby("cluster"):
    print(f"\nCluster {cluster_id}:")
    for doc in group["document"]:
        print("  -", doc)


Cluster 0:
  - How do I change a flat tire on a car?
  - How do I fix a leaky faucet?

Cluster 1:
  - What is the capital of France?
  - How do I bake a chocolate cake?
  - What is the distance between Earth and Mars?

Cluster 2:
  - What is the best way to learn Python?


In [17]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import adjusted_rand_score

# 4 clearly different topics, so we can check whether the clusters recover them
categories = ["sci.space", "rec.autos", "comp.graphics", "talk.politics.misc"]
news = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes"),   # avoid metadata leaking the topic
    random_state=42
)

texts = news.data[:1000]
true_labels = news.target[:1000]

news_emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

km_news = KMeans(n_clusters=4, random_state=42, n_init=10)
news_clusters = km_news.fit_predict(news_emb)

# ARI: 1 = clusters match the true topics perfectly, ~0 = no better than random
print("Adjusted Rand Index:", round(adjusted_rand_score(true_labels, news_clusters), 3))

true_names = [news.target_names[i] for i in true_labels]
print(pd.crosstab(pd.Series(true_names, name="true topic"),
                  pd.Series(news_clusters, name="cluster")))

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Adjusted Rand Index: 0.475
cluster               0    1   2    3
true topic                           
comp.graphics        15   14   7  232
rec.autos           209   28  16    3
sci.space           193   49   7   12
talk.politics.misc    4  205   6    0


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

### How many clusters make the most sense? Why?

| k | Silhouette score |
|---|---|
| 2 | 0.024 |
| 3 | **0.029** |
| 4 | 0.021 |
| 5 | 0.010 |

**k = 3 has the highest silhouette score**, so it's the best option among those tested, and it also gives the most interpretable groups (see below).

However, the most important finding is that **all the scores are very close to 0**. The silhouette score ranges from −1 to 1, and values near 0 mean that each document is about as close to the other clusters as to its own. In other words, **these 6 documents don't have a strong natural cluster structure**: they are 6 fairly unrelated questions (geography, cooking, astronomy, cars, programming, plumbing), so they sit at roughly similar distances from each other.

The difference between k = 3 (0.029) and k = 2 (0.024) is tiny, so the choice of k = 3 is supported more by interpretability than by the metric. With only 6 points, the silhouette score is also unstable: moving a single document between clusters can change it noticeably. Clustering is most useful when the data contains several documents per topic, which is why the larger dataset below gives much clearer results.

### Examine the documents in each cluster. Are they semantically meaningful? Can you assign a theme to each?

| Cluster | Documents | Theme |
|---|---|---|
| 0 | "How do I change a flat tire on a car?", "How do I fix a leaky faucet?" | **Practical repairs / DIY** |
| 1 | "What is the capital of France?", "What is the distance between Earth and Mars?", "How do I bake a chocolate cake?" | **General knowledge** (mixed) |
| 2 | "What is the best way to learn Python?" | **Learning / programming** |

- **Cluster 0 is clearly meaningful:** both documents are about fixing something physical at home or on a car. This is the most coherent group.
- **Cluster 1 is only partly meaningful:** France and Mars fit together as factual geography / astronomy questions, but the **chocolate cake doesn't belong there**. As the only cooking question, it has no natural partner, so it joined whichever cluster was closest. This is the "odd one out" effect: KMeans must assign every document to some cluster, even when it doesn't fit any of them.
- **Cluster 2 is a single document:** Python is the only technology / learning question, so it ends up alone.

**Topic or phrasing?** The clusters do **not** follow sentence structure: "How do I bake a chocolate cake?" was grouped with two "What is...?" questions rather than with the other "How do I...?" questions. The two repair questions are grouped together because of their *topic*, not only their phrasing. So in this case the model grouped mainly by meaning, though weakly, as the silhouette scores show.

### Try this exercise with a larger dataset of your choice

I used 1,000 posts from 4 different topics of the **20 Newsgroups** dataset (computer graphics, cars, space, politics). Because the true topic of every post is known, the clustering can be measured.

**Adjusted Rand Index = 0.475.** ARI compares the clusters with the true topics while ignoring the label numbers: 1 means a perfect match, ~0 means random. 0.475 means a **moderate match**, far better than random but far from perfect.

| True topic | Cluster 0 | Cluster 1 | Cluster 2 | Cluster 3 |
|---|---|---|---|---|
| comp.graphics | 15 | 14 | 7 | **232** |
| rec.autos | **209** | 28 | 16 | 3 |
| sci.space | **193** | 49 | 7 | 12 |
| talk.politics.misc | 4 | **205** | 6 | 0 |

**Interpretation:**
- **Computer graphics → cluster 3** and **politics → cluster 1** are recovered very well: each topic sends almost all its posts to one cluster.
- **Cars and space were merged into cluster 0** (209 + 193 posts). This is the main reason the ARI isn't higher. Both topics deal with vehicles, engines, speed and engineering, so their posts are likely close to each other in the embedding space.
- **Cluster 2 is very small** (36 posts in total, spread across all 4 topics). Because headers, footers and quotes were removed, some posts are probably left almost empty or very short, and these end up grouped together because they have no clear topic. Printing a few posts from cluster 2 would confirm this.
- **Cluster 1 also absorbed some cars (28) and space (49) posts**, probably posts touching on regulation, funding or policy, which overlap with political vocabulary.

**Conclusion:** with several documents per topic, the embeddings recover meaningful clusters. But KMeans with k = 4 doesn't guarantee one cluster per topic: it found one "junk" cluster of near-empty posts and had to merge two related topics to compensate. Removing empty / very short posts before clustering, or testing k = 5, would likely separate cars from space and raise the ARI.

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [18]:
import numpy as np
import pandas as pd

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Encode once: documents don't change between searches, only queries do
doc_embeddings = model.encode(documents, normalize_embeddings=True)

In [19]:
def semantic_search(query, documents, doc_embeddings, top_n=5):
    """Encode the query and return the top_n most semantically similar documents."""
    # 1. Encode the query the SAME way as the documents
    query_embedding = model.encode(query, normalize_embeddings=True)

    # 2. With length-1 vectors, the dot product IS the cosine similarity
    scores = doc_embeddings @ query_embedding

    # 3. Positions of the highest scores, best first
    top_idx = np.argsort(scores)[::-1][:top_n]

    # 4. Return (not print) a readable table, so the result can be reused
    return pd.DataFrame({
        "rank": range(1, len(top_idx) + 1),
        "score": scores[top_idx].round(3),
        "document": [documents[i] for i in top_idx]
    })

In [20]:
query = "Explain programming languages."
semantic_search(query, documents, doc_embeddings)

,rank,score,document
0,1,0.435,What is quantum computing?
1,2,0.319,What is the best way to learn Python?
2,3,0.110,How do I build a mobile app?
3,4,0.091,How do I set up a local server?
4,5,0.091,What are the best travel destinations in Europe?


In [21]:
for q in ["Explain programming languages.",
          "Where should I go on holiday?",
          "My car broke down",
          "Dessert recipes"]:
    print(f"\nQuery: {q}")
    print(semantic_search(q, documents, doc_embeddings, top_n=3).to_string(index=False))


Query: Explain programming languages.
 rank  score                              document
    1  0.435            What is quantum computing?
    2  0.319 What is the best way to learn Python?
    3  0.110          How do I build a mobile app?

Query: Where should I go on holiday?
 rank  score                                         document
    1  0.476 What are the best travel destinations in Europe?
    2  0.192                   What is the capital of France?
    3  0.161            What is the best way to learn Python?

Query: My car broke down
 rank  score                              document
    1  0.242 How do I change a flat tire on a car?
    2  0.178          How do I fix a leaky faucet?
    3  0.123          How do I build a mobile app?

Query: Dessert recipes
 rank  score                                         document
    1  0.495                  How do I bake a chocolate cake?
    2  0.123                     How do I build a mobile app?
    3  0.120 What are the best 

In [22]:
def semantic_search_explained(query, documents, doc_embeddings, top_n=5, min_score=0.2):
    results = semantic_search(query, documents, doc_embeddings, top_n)
    results = results[results["score"] >= min_score].copy()   # drop weak matches
    results["relevance"] = pd.cut(
        results["score"],
        bins=[-1, 0.3, 0.5, 1],
        labels=["Low", "Medium", "High"]
    )
    return results

semantic_search_explained("Explain programming languages.", documents, doc_embeddings)

,rank,score,document,relevance
0,1,0.435,What is quantum computing?,Medium
1,2,0.319,What is the best way to learn Python?,Medium


In [23]:
# Short previews so the results table stays readable
previews = [t.strip().replace("\n", " ")[:120] for t in texts]

for q in ["NASA rocket launch", "engine problems in cold weather", "3D rendering software"]:
    print(f"\nQuery: {q}")
    print(semantic_search(q, previews, news_emb, top_n=5).to_string(index=False))


Query: NASA rocket launch
 rank  score                                                                                                                 document
    1  0.411 There is a guy in NASA Johnson Space Center  that might answer  your question. I do not have his name right now but if y
    2  0.398 Speech by Pete Worden                                                Delivered Before the U.S. Space Foundation Conferen
    3  0.389 Could it be Public Missile, Inc in Michigan?  From the description of ad here, it sounds like they're talking about "Hig
    4  0.381 Reply address: mark.prado@permanet.org   > From: higgins@fnalf.fnal.gov (Bill Higgins-- Beam Jockey)  >  > In article <1
    5  0.381 COMMERCIAL SPACE NEWS/SPACE TECHNOLOGY INVESTOR NUMBER 22     This is number twenty-two in an irregular series on commer

Query: engine problems in cold weather
 rank  score                                                                                                                 docu

### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

### What are the top-ranked results for the given queries?

**Query: "Explain programming languages."**

| Rank | Score | Document |
|---|---|---|
| 1 | **0.435** | What is quantum computing? |
| 2 | **0.319** | What is the best way to learn Python? |
| 3 | 0.110 | How do I build a mobile app? |
| 4 | 0.091 | How do I set up a local server? |
| 5 | 0.091 | What are the best travel destinations in Europe? |

**Surprising result: quantum computing ranks above Python**, even though Python *is* a programming language. Likely reasons:

- **"Computing" is a strong concept match.** "Programming" and "computing" are closely related ideas, so the word itself pulls quantum computing up.
- **The intent matches.** "Explain X" asks for an explanation of a concept, and "What is quantum computing?" is exactly that kind of question. "What is the *best way to learn* Python?" asks about a learning method, which is a different intent.

This connects with Task 1: the model weighs **sentence intent and structure**, not only topic. The ideal answer for a user (Python) is still in the top 2.

**The score gaps tell the real story:** there is a big drop from rank 2 (0.319) to rank 3 (0.110), so only **2 documents are genuinely relevant**. Ranks 3–5 are near-noise, and rank 5 is a travel question about Europe with no connection to programming. It only appears because the function always returns 5 results.

**Extra queries (top result and gap to #2):**

| Query | Top result | Score | #2 score |
|---|---|---|---|
| "Where should I go on holiday?" | What are the best travel destinations in Europe? | 0.476 | 0.192 |
| "My car broke down" | How do I change a flat tire on a car? | 0.242 | 0.178 |
| "Dessert recipes" | How do I bake a chocolate cake? | 0.495 | 0.123 |

All three found the correct document **with zero shared keywords** (holiday → travel destinations, broke down → flat tire, dessert → chocolate cake). This is the key advantage of semantic search over keyword search.

- **Holiday and dessert** are clear matches: the top score is more than double the second.
- **"My car broke down"** is correct but weak (0.242), with a small gap to the leaky faucet (0.178). The model links "broke down" to *repairs* in general, so both repair questions score similarly. "Broke down" is also vaguer than "flat tire": many different problems can break a car down.

### How can you improve the ranking explanation for users?

A bare score like "0.435" means nothing to a user. Improvements, from simplest to most advanced:

1. **Relevance labels** ("High / Medium / Low") instead of raw numbers.
2. **A minimum-score threshold**, so the system returns fewer but relevant results instead of always padding the list to 5.
3. **Showing why a result matched**, for example highlighting shared words or a short generated explanation ("Matched on: computing").
4. **Re-ranking with a cross-encoder**: the fast bi-encoder retrieves candidates, and a slower but more accurate model that reads query and document *together* re-orders them. This could fix cases like quantum computing outranking Python.

The `semantic_search_explained()` function implements the first two. With `min_score=0.2`, it returned only **quantum computing (0.435)** and **Python (0.319)**, both labelled **Medium**. The threshold removed the three irrelevant results, so the user sees 2 useful answers instead of 5 mixed ones.

**Calibration lesson:** no result reached "High", even the correct ones. The best matches in all our tests scored between about 0.4 and 0.56, so cut-offs of 0.3 / 0.5 are too strict for this model. Thresholds must be calibrated by looking at real scores, because each model uses its own scale.

### Try this approach with a larger dataset

The same function was applied unchanged to the 1,000 **20 Newsgroups** posts from Task 2.

| Query | Score range (top 5) | Results |
|---|---|---|
| "NASA rocket launch" | 0.381 – 0.411 | All 5 about space: NASA Johnson Space Center, a Space Foundation speech, a missile company, commercial space news |
| "engine problems in cold weather" | 0.261 – 0.382 | Ranks 1–4 about car engines (oil drain valve, winter driving, oil viscosity, engine components); **rank 5 is about rocket thruster tanks** |
| "3D rendering software" | 0.523 – 0.564 | All 5 about 3D graphics software and libraries |

**Interpretation:**
- **"3D rendering software"** got the highest and most consistent scores. Computer graphics posts use very specific vocabulary, so the topic is easy to identify.
- **"NASA rocket launch"** returned 5 space posts with very close scores (0.381–0.411). Many posts are similarly relevant, so the exact order among them matters little.
- **"engine problems in cold weather"** shows a **cross-topic error**: rank 5 is a space post about *cold gas* thrusters for rockets. The model matched "cold" + engine-like hardware, the same confusion between cars and space that made KMeans merge those two topics in Task 2.

**Conclusion:** semantic search works well on real, messy text, finding relevant posts without keyword matching. Its errors happen where topics share vocabulary (cars and rockets both have engines, fuel and cold-weather issues).

**Scaling note:** a matrix product over 1,000 documents is instant, but with millions of documents comparing against everything becomes slow. Vector databases and approximate nearest-neighbour libraries such as **FAISS** solve this by finding the closest vectors without checking every one.